# MEN 4338 · Week 5 — Variogram models and fitting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W5_variogram_models_fitting.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

The experimental variogram is only known at a few lags, and kriging needs $\gamma(h)$ at **every** distance
and direction. We therefore fit a **model**: a smooth function from a small family of shapes that are
mathematically safe (they never give a negative kriging variance). This week ends with the model of the
Bassam deposit that we will use for estimation from week 8.

**What you will practise:** nugget, spherical, exponential and Gaussian models · practical range ·
nested structures · weighted least-squares fitting · anisotropic models.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w5/q1.py', 'tests/q1.py'), ('tests/w5/q2.py', 'tests/q2.py'), ('tests/w5/q3.py', 'tests/q3.py'), ('tests/w5/q4.py', 'tests/q4.py'), ('tests/w5/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def exp_variogram(x, y, v, lag, nlags, azimuth=None, atol=22.5):
    """Experimental semivariogram. Lag class k (1..nlags) holds the pairs with |d - k*lag| < lag/2.
    With an azimuth (degrees clockwise from north), only pairs within +/- atol of that direction are kept.
    Returns mean distance, semivariance and number of pairs of each class (NaN where a class is empty)."""
    x, y, v = (np.asarray(a, float) for a in (x, y, v))
    i, j = np.triu_indices(len(v), 1)
    dx, dy = x[j] - x[i], y[j] - y[i]
    d = np.hypot(dx, dy)
    keep = np.ones(len(d), bool)
    if azimuth is not None:
        az = np.degrees(np.arctan2(dx, dy)) % 180
        diff = np.abs(az - azimuth % 180)
        keep = np.minimum(diff, 180 - diff) <= atol
    sq = 0.5 * (v[j] - v[i]) ** 2
    hm, g, n = np.full(nlags, np.nan), np.full(nlags, np.nan), np.zeros(nlags, int)
    for k in range(1, nlags + 1):
        m = keep & (np.abs(d - k * lag) < lag / 2)
        n[k - 1] = m.sum()
        if n[k - 1]:
            hm[k - 1], g[k - 1] = d[m].mean(), sq[m].mean()
    return hm, g, n

## 1. The standard models

| model | $\gamma(h)$ for $h > 0$ | behaviour near 0 |
|---|---|---|
| nugget | $c_0$ | discontinuous |
| spherical | $c_0 + c\left(\tfrac32 \tfrac{h}{a} - \tfrac12 \tfrac{h^3}{a^3}\right)$ if $h < a$, else $c_0 + c$ | linear: most grades |
| exponential | $c_0 + c\,(1 - e^{-3h/a})$ | linear, reaches the sill only asymptotically |
| Gaussian | $c_0 + c\,(1 - e^{-3h^2/a^2})$ | parabolic: very smooth variables (topography, thickness) |

and $\gamma(0) = 0$ always. For the exponential and Gaussian models $a$ is the **practical range**: the
distance where 95 % of the sill is reached.

Several structures can be added (**nested** model), e.g. nugget + short-range spherical + long-range spherical.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

h = np.linspace(0, 400, 401)
r = np.minimum(h / 200, 1)
sph = np.where(h > 0, 0.05 + 0.2 * (1.5 * r - 0.5 * r ** 3), 0)
exp_ = np.where(h > 0, 0.05 + 0.2 * (1 - np.exp(-3 * h / 200)), 0)
gau = np.where(h > 0, 0.05 + 0.2 * (1 - np.exp(-3 * (h / 200) ** 2)), 0)
plt.figure(figsize=(7, 4))
for g, lab in [(sph, "spherical"), (exp_, "exponential"), (gau, "Gaussian")]:
    plt.plot(h, g, label=lab)
plt.axvline(200, ls=":", color="grey"); plt.axhline(0.25, ls="--", color="grey")
plt.xlabel("h (m)"); plt.ylabel("γ(h)"); plt.legend(); plt.title("c0 = 0.05, c = 0.2, a = 200 m")
plt.show()

## 2. Fitting

Fitting "by eye" is still common, and a good habit to check any automatic fit. An automatic fit minimises the
**weighted sum of squared errors**, giving more weight to lags with many pairs:

$$WSSE(c_0, c, a) = \sum_k N(h_k)\,\big(\hat\gamma(h_k) - \gamma_{model}(h_k)\big)^2$$

Always look at the result: a perfect numerical fit with a silly range is still silly.

In [ ]:
samples = pd.read_csv("bassam_samples.csv")
grid = samples[samples["campaign"] == "grid"]
x, y, au = grid["x"].to_numpy(), grid["y"].to_numpy(), grid["au_gpt"].to_numpy()

## 3. Anisotropic models

A geometric anisotropy is modelled by **rescaling distances**: rotate the separation vector into the major
($u$) and minor ($w$) axes, stretch the minor component by $a_{max}/a_{min}$, and apply an isotropic model
with range $a_{max}$:

$$u = \Delta x \sin\theta + \Delta y \cos\theta, \quad w = \Delta x \cos\theta - \Delta y \sin\theta, \quad
h' = \sqrt{u^2 + \left(w \, a_{max}/a_{min}\right)^2}$$

with $\theta$ the azimuth of the major direction.

---
## Exercises

### Q1. The spherical model

Write `spherical(h, c0, c, a)`, working on numbers and arrays, with $\gamma(0) = 0$.

In [ ]:
def spherical(h, c0, c, a):
    ...

spherical(np.array([0, 50, 100, 200, 300]), 0.05, 0.2, 200)

In [ ]:
grader.check("q1")

### Q2. Exponential and Gaussian

Write `exponential(h, c0, c, a)` and `gaussian(h, c0, c, a)` with the practical-range convention, and
$\gamma(0) = 0$.

In [ ]:
def exponential(h, c0, c, a):
    ...

def gaussian(h, c0, c, a):
    ...

exponential(200.0, 0, 1, 200), gaussian(200.0, 0, 1, 200)

In [ ]:
grader.check("q2")

### Q3. The fitting criterion

Compute the experimental variogram of the grid holes, lag 50 m and 12 lags: `h_exp, g_exp, n_exp`.
Write `wsse(params, h, g, n)` returning the weighted sum of squared errors of a spherical model with
`params = (c0, c, a)`.

In [ ]:
h_exp, g_exp, n_exp = ...

def wsse(params, h, g, n):
    ...

wsse((0.05, 0.2, 250), h_exp, g_exp, n_exp)

In [ ]:
grader.check("q3")

### Q4. Best spherical model

Search all combinations of $c_0$ from 0 to 0.15 (step 0.01), $c$ from 0.05 to 0.30 (step 0.01) and $a$
from 50 to 500 m (step 10) and keep the one with the smallest `wsse` in `best_params = (c0, c, a)`.
Store `nugget_ratio` = $c_0 / (c_0 + c)$. Plot the experimental points and the fitted model.

*Hint:* `np.arange(0, 0.151, 0.01)` includes 0.15; three nested `for` loops are fine (about 19 000 tries).

In [ ]:
best_params = ...
...
nugget_ratio = ...
best_params, nugget_ratio

In [ ]:
grader.check("q4")

### Q5. The Bassam model

Write `aniso_h(dx, dy, azimuth, a_major, a_minor)` returning the rescaled distance $h'$. Then define
`gamma_bassam(dx, dy)`: the model fitted on the directional variograms of week 4, which we keep for the rest
of the semester:

* nugget 0.06, one spherical structure with partial sill 0.20,
* major direction N045 with range 300 m, minor direction N135 with range 150 m.

Check it: plot the model along N045 and N135 on top of the directional variograms.

In [ ]:
def aniso_h(dx, dy, azimuth, a_major, a_minor):
    ...

def gamma_bassam(dx, dy):
    ...

gamma_bassam(np.array([70.7, -70.7]), np.array([70.7, 70.7]))   # 100 m along N045 and along N135

In [ ]:
grader.check("q5")

### Going further (not checked)

* Use `scipy.optimize.curve_fit` with `sigma=1/np.sqrt(n_exp)` to fit the spherical model. Same answer?
* Fit an exponential model and compare its WSSE with the spherical one. Which one would you choose, and why?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()